# Рекомендация тарифов

***Описание проекта***

    Оператор мобильной связи «Мегалайн» выяснил: многие клиенты пользуются архивными тарифами. Они хотят построить систему, способную проанализировать поведение клиентов и предложить пользователям новый тариф: «Смарт» или «Ультра».
    
    В вашем распоряжении данные о поведении клиентов, которые уже перешли на эти тарифы (из проекта курса «Статистический анализ данных»). Нужно построить модель для задачи классификации, которая выберет подходящий тариф. Предобработка данных не понадобится — вы её уже сделали.
    
    Постройте модель с максимально большим значением accuracy. Чтобы сдать проект успешно, нужно довести долю правильных ответов по крайней мере до 0.75. Проверьте accuracy на тестовой выборке самостоятельно.

***Описание данных***

    Каждый объект в наборе данных — это информация о поведении одного пользователя за месяц. Известно:

    сalls — количество звонков,
    minutes — суммарная длительность звонков в минутах,
    messages — количество sms-сообщений,
    mb_used — израсходованный интернет-трафик в Мб,
    is_ultra — каким тарифом пользовался в течение месяца («Ультра» — 1, «Смарт» — 0).

## Откройте и изучите файл

In [1]:
#импорт библиотек 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import joblib

In [2]:
users = pd.read_csv('/datasets/users_behavior.csv') #читаем файл и сохраняем его в переменной users

Посмотрим на информацию о таблице 

In [3]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
calls       3214 non-null float64
minutes     3214 non-null float64
messages    3214 non-null float64
mb_used     3214 non-null float64
is_ultra    3214 non-null int64
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


Замечаем, что нужно поменять тип данных столбцов calls (количество звонков) и messages () на целочисленный

In [4]:
users['calls'] = users['calls'].astype('int')

In [5]:
users['messages'] = users['messages'].astype('int')

Проверим

In [6]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
calls       3214 non-null int64
minutes     3214 non-null float64
messages    3214 non-null int64
mb_used     3214 non-null float64
is_ultra    3214 non-null int64
dtypes: float64(2), int64(3)
memory usage: 125.7 KB


Проверим таблицу на наличие дубликатов и пропусков

In [7]:
users.duplicated().sum()

0

In [8]:
users.isna().sum()

calls       0
minutes     0
messages    0
mb_used     0
is_ultra    0
dtype: int64

Дубликатов и пропусков - нет

Посмотрим первые 5 строк таблицы

In [9]:
users.head()

,calls,minutes,messages,mb_used,is_ultra
0,40,311.90,83,19915.42,0
1,85,516.75,56,22696.96,0
2,77,467.66,86,21060.45,0
3,106,745.53,81,8437.39,1
4,66,418.74,1,14502.75,0


***Пропусков и дубликатов нет, мы взглянули на таблицу и информацию о ней, заменили тип данных столбцов calls и messages на целочисленный.***

## Разбейте данные на выборки

Разобьём данные на три выборки (обучающую, валидационную, тестовую)

In [10]:
train_users, valid_users = train_test_split(users, test_size=0.3, random_state=12345)
valid_users, test_users = train_test_split(valid_users, test_size = 0.5, random_state=12345)

In [11]:
print('Обучающая выборка:')
print(train_users.shape[0],'объектов')
print('___________________________________________________')
print('Валидационная выборка:')
print(valid_users.shape[0],'объектов')
print('___________________________________________________')
print('Тестовая выборка:')
print(test_users.shape[0],'объектов')

Обучающая выборка:
2249 объектов
___________________________________________________
Валидационная выборка:
482 объектов
___________________________________________________
Тестовая выборка:
483 объектов


## Исследуйте модели

Запишем в переменные целевой признак и признаки

In [12]:
features = users.drop(['is_ultra'], axis=1)
target = users['is_ultra']

features_valid = valid_users.drop(['is_ultra'], axis=1)
target_valid = valid_users['is_ultra']

features_train = train_users.drop(['is_ultra'], axis=1)
target_train = train_users['is_ultra']

features_test = test_users.drop(['is_ultra'], axis=1)
target_test = test_users['is_ultra']

**Модель "Дерево решений", проверим разные глубины. Найдём accuracy наилучшей модели.**

In [13]:
best_result=0
best_depth=0
for depth in range(1,16):
    model = DecisionTreeClassifier(random_state=12345, max_depth=depth)
    model.fit(features_train, target_train)
    predictions = model.predict(features_valid)
    result = accuracy_score(target_valid, predictions)
    print('Depth:', depth, 'Accuracy:', result)
    if result > best_result:
        best_depth=depth
        best_result=result
print('_______________________________________')
print()
print('Accuracy наилучшей модели:',best_result,'( при depth =',best_depth,')')


Depth: 1 Accuracy: 0.7510373443983402
Depth: 2 Accuracy: 0.7800829875518672
Depth: 3 Accuracy: 0.7863070539419087
Depth: 4 Accuracy: 0.7883817427385892
Depth: 5 Accuracy: 0.7614107883817427
Depth: 6 Accuracy: 0.7842323651452282
Depth: 7 Accuracy: 0.7821576763485477
Depth: 8 Accuracy: 0.7821576763485477
Depth: 9 Accuracy: 0.7780082987551867
Depth: 10 Accuracy: 0.7883817427385892
Depth: 11 Accuracy: 0.7904564315352697
Depth: 12 Accuracy: 0.7946058091286307
Depth: 13 Accuracy: 0.7821576763485477
Depth: 14 Accuracy: 0.7821576763485477
Depth: 15 Accuracy: 0.7593360995850622
_______________________________________

Accuracy наилучшей модели: 0.7946058091286307 ( при depth = 12 )


**Модель "Случайный лес", проверим разные количества деревьев в лесу/разные глубины. Найдём accuracy наилучшей модели**

In [14]:
maxim_est=0
maxim_depth=0
classes = 
for est in range(20,31,2):
    for depth in range(1,11):
        for x in classes:
            model =  model_forest = RandomForestClassifier(
                class_weight= x, max_features = p[2], max_depth = p[3], 
                n_estimators = 10, random_state=12345)
            model.fit(features_train, target_train)
            predictions = model.predict(features_valid)
            result = accuracy_score(target_valid, predictions)
            print('Estimator:', est,'Depth:', depth, 'Accuracy:', result)
            if result > maxim_result:
                maxim_result=result
                maxim_depth=depth
                maxim_est=est
print('____________________________________________________')
print()
print('Наилучшей модели:','( при depth =',maxim_depth,'и estimator =',maxim_est,'и  =',maxim_est,' ,' )')


Estimator: 20 Depth: 1 Accuracy: 0.7510373443983402
Estimator: 20 Depth: 2 Accuracy: 0.7925311203319502
Estimator: 20 Depth: 3 Accuracy: 0.7904564315352697
Estimator: 20 Depth: 4 Accuracy: 0.7966804979253111
Estimator: 20 Depth: 5 Accuracy: 0.7946058091286307
Estimator: 20 Depth: 6 Accuracy: 0.7966804979253111
Estimator: 20 Depth: 7 Accuracy: 0.8029045643153527
Estimator: 20 Depth: 8 Accuracy: 0.7987551867219918
Estimator: 20 Depth: 9 Accuracy: 0.7966804979253111
Estimator: 20 Depth: 10 Accuracy: 0.8091286307053942
Estimator: 22 Depth: 1 Accuracy: 0.7551867219917012
Estimator: 22 Depth: 2 Accuracy: 0.7883817427385892
Estimator: 22 Depth: 3 Accuracy: 0.7904564315352697
Estimator: 22 Depth: 4 Accuracy: 0.7966804979253111
Estimator: 22 Depth: 5 Accuracy: 0.7966804979253111
Estimator: 22 Depth: 6 Accuracy: 0.7946058091286307
Estimator: 22 Depth: 7 Accuracy: 0.8008298755186722
Estimator: 22 Depth: 8 Accuracy: 0.7966804979253111
Estimator: 22 Depth: 9 Accuracy: 0.8008298755186722
Estimator: 

**Модель логической регрессии. Найдём accuracy в зависимости от параметра solver, найдем наилучшую.**

In [15]:
model = LogisticRegression(random_state=12345,solver='lbfgs') 
model.fit(features_train, target_train)
score = model.score(features_valid,target_valid) 
print("Accuracy модели логистической регрессии на валидационной выборке при solver='lbfgs':", score)

Accuracy модели логистической регрессии на валидационной выборке при solver='lbfgs': 0.6908713692946058


In [16]:
model = LogisticRegression(random_state=12345,solver='liblinear') 
model.fit(features_train, target_train)
result = model.score(features_valid,target_valid) 
print("Accuracy модели логистической регрессии на валидационной выборке при solver='liblinear':", result)

Accuracy модели логистической регрессии на валидационной выборке при solver='liblinear': 0.6950207468879668


In [17]:
if result>score:
    print('Accuracy наилучшей модели:',result,'при solver=liblinear')
else:
    print('Accuracy наилучшей модели:',score,'при solver=lbfgs')

Accuracy наилучшей модели: 0.6950207468879668 при solver=liblinear


***Исследовали три модели : Случайный лес, Дерево решений, Логическая регрессия. Наилучшие модели:***
   
    У дерева решений при depth 12 accuracy: 0.7946058091286307
    У случайного леса при estimator 24, depth 10 accuracy: 0.8132780082987552
    У логической регрессии при solver=liblinear  accuracy: 0.6950207468879668
    
***Таким образом, мы понимаем, что модель "Случайный лес" показывает лучший результат (accuracy: 0.8132780082987552 при estimator 24, depth 10)***

## Проверьте модель на тестовой выборке

Проверим модель "Случайный лес" при depth=10, estimator=24 на тестовой выборке.

In [18]:
model = RandomForestClassifier(random_state=12345, n_estimators=40, max_depth=10)
model.fit(features_train,target_train)
predictions = model.predict(features_test)
result = accuracy_score(target_test, predictions)
print('Модель при обучении на тестовой выборке показала результат accuracy', result)

Модель при обучении на тестовой выборке показала результат accuracy 0.8074534161490683


## (бонус) Проверьте модели на адекватность

Посмотрим в процентах распределение клиентов по тарифам 

In [19]:
users['is_ultra'].value_counts(normalize=True)

0    0.693528
1    0.306472
Name: is_ultra, dtype: float64

Примерно 69% клиентов пользуются тарифом «Смарт», а около 30% - тарифом "Ультра".

*Пусть наша модель всегда будет предсказывать тариф «Смарт» - точность модели будет равна примерно 69%, но точность нашей лучшей модели больше и равна чуть больше 80%. Таким образом, наша модель адекватная, не предсказывает случайным образом.*